# Prometheus v1 — Colab Runtime Template

**Do not edit this notebook manually.** The harness layer may be updated by
the governor-approved Ashby loop; the gate and checkpoint cells are protected.

## How a session works
1. **Cell 1 – Config**: set `RUN_ID`, `AUDIT_SEEDS`, and `MODEL_PINS`.
2. **Cell 2 – Mount & install**: mounts Drive, installs `prometheus-v1` from
   the pinned wheel on Drive, initialises the persistent Gate.
3. **Cell 3 – Gate check**: verifies the heartbeat; aborts if the governor
   has halted the run.
4. **Cell 4 – Run loop**: sandboxed episode loop, checkpointing to Drive after
   every episode so a forced restart resumes cleanly.
5. **Cell 4b – Foundation-model bake-off**: 200-episode evaluation with
   heartbeat checks and local SQLite MLflow DB snapshotted to Drive.
6. **Cell 5 / 6 – Propose & install variant**: packages harness bundles with
   `write_bundle()` + `.ready` marker and installs governor-approved variants.

After a Colab forced restart, re-run from **Cell 2** (Drive stays mounted,
gate state and checkpoint resume automatically).

---
## Cell 1 — Configuration *(edit this cell)*

In [ ]:
# ── User-editable configuration ───────────────────────────────────────────────
RUN_ID = "run-001"          # must match the governor's policy run_id

# Must match the governor's policy.json (audit_seeds and model_pins)
AUDIT_SEEDS = list(range(1000, 1050))
MODEL_PINS  = {}            # e.g. {"cortex": "qwen3-8b@<hf-commit>", "prover": "goedel-v2-8b@<hf-commit>"}

# Google Drive paths (relative to MyDrive root)
DRIVE_ROOT      = "/content/drive/MyDrive"
PROMETHEUS_DIR  = f"{DRIVE_ROOT}/prometheus"          # gdrive:prometheus
EXCHANGE_DIR    = f"{PROMETHEUS_DIR}/{RUN_ID}"         # gdrive:prometheus/run-001
PUB_KEY_PATH    = f"{PROMETHEUS_DIR}/governor_public_key.txt"
WHEEL_GLOB      = f"{PROMETHEUS_DIR}/prometheus_v1-*.whl"  # pre-built wheel

# Persistent paths on Drive so installed variants and gate state survive Colab restarts
INSTALL_ROOT    = f"{EXCHANGE_DIR}/installed_harness"
GATE_STATE_PATH = f"{EXCHANGE_DIR}/gate_state.json"
CHECKPOINT_PATH = f"{EXCHANGE_DIR}/checkpoint.json"

# Heartbeat tolerance (should match governor policy: heartbeat_ttl_s = 900)
MAX_CLOCK_SKEW_S = 60.0

print(f"RUN_ID        : {RUN_ID}")
print(f"Exchange dir  : {EXCHANGE_DIR}")
print(f"Public key    : {PUB_KEY_PATH}")
print(f"Install root  : {INSTALL_ROOT}")
print(f"Gate state    : {GATE_STATE_PATH}")
print(f"Checkpoint    : {CHECKPOINT_PATH}")

---
## Cell 2 — Mount Drive & install package
*Re-run this after every forced restart.*

In [ ]:
import os, sys, json, time, glob, shutil, hashlib, subprocess, pathlib

# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Drive mounted.")

# ── 2. Verify exchange directory and public key exist ─────────────────────────
for p in [EXCHANGE_DIR, PUB_KEY_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"Expected path not found: {p}\n"
            "Check that the governor has started and rclone has synced."
        )
print(f"Exchange dir  : OK  ({EXCHANGE_DIR})")

GOV_PUB_KEY_HEX = pathlib.Path(PUB_KEY_PATH).read_text().strip()
assert len(GOV_PUB_KEY_HEX) == 64, "governor_public_key.txt should be a 64-char hex string"
print(f"Governor key  : {GOV_PUB_KEY_HEX[:16]}…  (loaded)")

# ── 3. Install prometheus-v1 ──────────────────────────────────────────────────
# Prefer a pre-built wheel uploaded to Drive (pinned, offline-safe).
wheels = sorted(glob.glob(WHEEL_GLOB))
if wheels:
    wheel = wheels[-1]   # newest
    print(f"Installing from wheel: {os.path.basename(wheel)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", wheel])
else:
    raise FileNotFoundError(
        f"No wheel matching {WHEEL_GLOB} found on Drive.\n"
        "Build one locally (`python -m build --wheel`) and upload dist/*.whl to Drive."
    )

# ── 4. Initialise the persistent Gate ─────────────────────────────────────────
from prometheus.runtime.gate import Gate, Halted, InstallRefused

baseline_hash_path = pathlib.Path(EXCHANGE_DIR) / "baseline_hash.txt"
if not baseline_hash_path.exists():
    raise FileNotFoundError(
        f"{baseline_hash_path} not found.\n"
        "Run `prometheus-governor init` and sync baseline_hash.txt from the workstation first."
    )
BASELINE_HASH = baseline_hash_path.read_text().strip()
print(f"Baseline hash : {BASELINE_HASH[:16]}…")

gate = Gate(
    governor_public_key_hex=GOV_PUB_KEY_HEX,
    run_id=RUN_ID,
    exchange=pathlib.Path(EXCHANGE_DIR),
    install_root=pathlib.Path(INSTALL_ROOT),
    baseline_hash=BASELINE_HASH,
    max_clock_skew_s=MAX_CLOCK_SKEW_S,
    state_path=pathlib.Path(GATE_STATE_PATH),
)
print(f"Gate ready    : generation={gate.generation}, "
      f"installed={gate.installed_hash[:16]}…")

---
## Cell 3 — Gate check
Run once after Cell 2. Cells 4 and 4b call this before every episode
automatically; this cell is here so you can test it manually.

In [ ]:
try:
    hb = gate.check_heartbeat()
    remaining = hb["expires_at"] - time.time()
    print(f"Heartbeat OK  : counter={hb['counter']}, "
          f"expires in {remaining:.0f}s, halt={hb.get('halt', False)}")
except Halted as e:
    print(f"HALTED: {e}")
    raise

---
## Cell 4 — Run loop
Runs Hidden-Law Lab episodes, checkpoints after each one.
Safe to interrupt and restart — it resumes from the checkpoint.

In [ ]:
from prometheus.lab import HiddenLawWorld
from prometheus.lab.laws import LAW_NAMES
from prometheus.runtime.sandbox import SandboxViolation, run as sandbox_run

# ── Checkpoint helpers ────────────────────────────────────────────────────────

def load_checkpoint() -> dict:
    """Load checkpoint from Drive, or return a fresh state."""
    p = pathlib.Path(CHECKPOINT_PATH)
    if p.exists():
        state = json.loads(p.read_text())
        print(f"Checkpoint resumed: episode {state['next_episode']}, "
              f"{len(state['results'])} results so far")
        return state
    print("No checkpoint found — starting fresh.")
    return {"next_episode": 0, "results": {}}


def save_checkpoint(state: dict) -> None:
    """Atomically write checkpoint to Drive."""
    p = pathlib.Path(CHECKPOINT_PATH)
    tmp = p.with_name(f".{p.name}.tmp")
    tmp.write_text(json.dumps(state, indent=2, sort_keys=True))
    tmp.replace(p)


# ── Episode configuration ─────────────────────────────────────────────────────

TOTAL_EPISODES = 200    # target for Phase 0 bake-off
NOISE          = 0.05
BUDGET         = 500.0

# Seeds: deterministic range so the governor can reproduce any world.
EPISODE_SEEDS  = list(range(TOTAL_EPISODES))

# ── Harness: run agent code inside the sandbox ────────────────────────────────
# Agent code NEVER receives ground-truth metadata (such as world.family).

AGENT_CODE = """
# Placeholder harness: claims all laws are False.
# Replace with model-generated strategy code in Phase 1.
return_value = {name: False for name in inputs["law_names"]}
"""


def run_episode_sandboxed(world: HiddenLawWorld) -> dict[str, bool]:
    """Run AGENT_CODE in the sandbox and return law claims {law_name: bool}."""
    world_inputs = {
        "law_names": list(LAW_NAMES),
        "symbols":   list(world.symbols),
        "n_symbols": len(world.symbols),
        "budget":    world.budget,
        "noise":     world.noise,
    }
    try:
        result = sandbox_run(
            AGENT_CODE,
            inputs=world_inputs,
            cpu_seconds=10,
            wall_seconds=30,
            memory_mb=256,
        )
        if result.error:
            print(f"  [sandbox error] {result.error[:120]}")
            return {name: False for name in LAW_NAMES}
        claims = result.return_value
        if not isinstance(claims, dict):
            return {name: False for name in LAW_NAMES}
        return {name: bool(claims.get(name, False)) for name in LAW_NAMES}
    except SandboxViolation as exc:
        print(f"  [sandbox violation] {exc}")
        return {name: False for name in LAW_NAMES}


# ── Run loop ──────────────────────────────────────────────────────────────────

state = load_checkpoint()

for episode_idx in range(state["next_episode"], TOTAL_EPISODES):
    seed = EPISODE_SEEDS[episode_idx]

    # Gate check before every episode: halts if the governor pulled the kill switch.
    try:
        gate.check_heartbeat()
    except Halted as e:
        save_checkpoint(state)
        print(f"\nRun halted by governor at episode {episode_idx}: {e}")
        raise SystemExit(0)

    world = HiddenLawWorld(seed=seed, noise=NOISE, budget=BUDGET)
    claims = run_episode_sandboxed(world)
    truth  = world.ground_truth()
    correct = sum(claims.get(k) == v for k, v in truth.items())

    state["results"][str(seed)] = {
        "claims" : claims,
        "truth"  : truth,
        "correct": correct,
        "family" : world.family,
        "n"      : len(world.symbols),
        "spent"  : world.spent,
    }
    state["next_episode"] = episode_idx + 1
    save_checkpoint(state)

    if (episode_idx + 1) % 10 == 0 or episode_idx == TOTAL_EPISODES - 1:
        done = state["next_episode"]
        avg  = sum(r["correct"] for r in state["results"].values()) / (done * len(LAW_NAMES))
        print(f"Episode {done:4d}/{TOTAL_EPISODES}  accuracy={avg:.3f}")

print("\nAll episodes complete.")
total = len(state["results"])
avg   = sum(r["correct"] for r in state["results"].values()) / (total * len(LAW_NAMES))
print(f"Final accuracy: {avg:.3f}  ({total} worlds × {len(LAW_NAMES)} laws)")

---
## Cell 4b — Foundation-model bake-off *(Phase 0, step 5)*

Runs 200 Hidden-Law Lab episodes against the selected cortex model, checking
the governor heartbeat on every episode and logging metrics to a local SQLite
MLflow DB (`/content/mlflow.db`) that is atomically snapshotted to Drive
(avoiding Google Drive FUSE SQLite lock/corruption issues).

**Prerequisites**
- vLLM is serving the model on `localhost:8000` (see Cell 4b-setup).
- Drive is mounted and the gate is initialised (Cells 2–3).

In [ ]:
# ── Cell 4b-setup: install extras and optionally start vLLM ──────────────────
import subprocess, sys
from prometheus.bakeoff import MODEL_REGISTRY

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "openai>=1.0", "mlflow>=2.0"])

# Uncomment to launch vLLM using the registry's T4-safe flags:
# MODEL_NAME = "qwen3-8b"  # or "gpt-oss-20b", "gemma4-4b"
# spec = MODEL_REGISTRY[MODEL_NAME]
# cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
#        "--model", spec.hf_id, *spec.vllm_flags]
# print("Launching:", " ".join(cmd))
# vllm_proc = subprocess.Popen(cmd, stdout=open("/tmp/vllm.log", "wb"), stderr=subprocess.STDOUT)
# print(f"vLLM PID: {vllm_proc.pid}  — tail /tmp/vllm.log until startup completes.")

print("Setup complete. When vLLM is ready, run Cell 4b-run.")

In [ ]:
# ── Cell 4b-run: execute the bake-off for one model ──────────────────────────
import pathlib, shutil
from prometheus.bakeoff import BakeoffConfig, BakeoffRunner, MODEL_REGISTRY
from prometheus.runtime.gate import Halted

MODEL_NAME       = "qwen3-8b"   # options: "qwen3-8b", "gpt-oss-20b", "gemma4-4b"
LOCAL_MLFLOW_DB  = pathlib.Path("/content/mlflow.db")
DRIVE_MLFLOW_DB  = pathlib.Path(f"{PROMETHEUS_DIR}/mlflow.db")
CKPT_PATH        = f"{PROMETHEUS_DIR}/bakeoff_{MODEL_NAME}.json"
VLLM_URL         = "http://localhost:8000/v1"

# Restore prior MLflow DB from Drive to local SSD (never write SQLite directly over Drive FUSE)
if DRIVE_MLFLOW_DB.exists() and not LOCAL_MLFLOW_DB.exists():
    shutil.copy2(DRIVE_MLFLOW_DB, LOCAL_MLFLOW_DB)

def snapshot_mlflow_db() -> None:
    if LOCAL_MLFLOW_DB.exists():
        tmp = DRIVE_MLFLOW_DB.with_name(f".{DRIVE_MLFLOW_DB.name}.tmp")
        shutil.copy2(LOCAL_MLFLOW_DB, tmp)
        tmp.replace(DRIVE_MLFLOW_DB)

# Gate check before starting
gate.check_heartbeat()

MLFLOW_URI = f"sqlite:///{LOCAL_MLFLOW_DB}"
print(f"Model      : {MODEL_NAME}  ({MODEL_REGISTRY[MODEL_NAME].hf_id})")
print(f"MLflow URI : {MLFLOW_URI}  (snapshotted to {DRIVE_MLFLOW_DB})")
print(f"Checkpoint : {CKPT_PATH}")
print()

config = BakeoffConfig(
    model_name=MODEL_NAME,
    n_episodes=200,
    noise=0.05,
    budget=500.0,
    vllm_base_url=VLLM_URL,
    temperature=0.0,
    max_tokens=2048,
    mlflow_tracking_uri=MLFLOW_URI,
    mlflow_experiment="phase0-bakeoff",
    checkpoint_path=CKPT_PATH,
)

runner = BakeoffRunner(config)

def _on_episode(episode_idx, result):
    # Enforce governor heartbeat on every episode
    try:
        gate.check_heartbeat()
    except Halted as exc:
        snapshot_mlflow_db()
        print(f"\nBake-off halted by governor at episode {episode_idx + 1}: {exc}")
        raise SystemExit(0)
    if (episode_idx + 1) % 20 == 0 or result.error:
        snapshot_mlflow_db()
        tag = f"[ERROR: {result.error[:60]}]" if result.error else ""
        print(f"  Episode {episode_idx+1:4d}/200  "
              f"family={result.family:20s}  acc={result.accuracy:.3f}  "
              f"lat={result.latency_s:.1f}s  {tag}")

try:
    summary = runner.run(on_episode=_on_episode)
finally:
    snapshot_mlflow_db()

print()
print(f"═══ Bake-off complete: {MODEL_NAME} ═══")
print(f"Accuracy       : {summary.accuracy:.3f}  ({summary.n_errors} errors)")
print(f"Mean latency   : {summary.mean_latency_s:.2f}s / episode")
print()
print("Per-law accuracy:")
for law, acc in sorted(summary.per_law_accuracy.items()):
    bar = "█" * int(acc * 20)
    print(f"  {law:20s}  {acc:.3f}  {bar}")
print()
print("Per-family accuracy:")
for fam, acc in sorted(summary.per_family_accuracy.items()):
    bar = "█" * int(acc * 20)
    print(f"  {fam:22s}  {acc:.3f}  {bar}")

In [ ]:
# ── Cell 4b-compare: print the leaderboard across all completed bake-off runs ─
import json, pathlib
from prometheus.bakeoff import MODEL_REGISTRY
from prometheus.lab.laws import LAW_NAMES

CANDIDATES = ["qwen3-8b", "gpt-oss-20b", "gemma4-4b"]
rows = []

for name in CANDIDATES:
    ckpt = pathlib.Path(f"{PROMETHEUS_DIR}/bakeoff_{name}.json")
    if not ckpt.exists():
        print(f"  {name}: no checkpoint yet")
        continue
    state = json.loads(ckpt.read_text())
    results = list(state["results"].values())
    if not results:
        continue
    n = len(results)
    total_correct = sum(
        0 if r.get("error") else sum(r["claims"].get(k) == r["truth"].get(k) for k in LAW_NAMES)
        for r in results
    )
    acc = total_correct / (n * len(LAW_NAMES))
    n_errors = sum(1 for r in results if r.get("error"))
    lats = [r.get("latency_s", 0) for r in results]
    mean_lat = sum(lats) / len(lats) if lats else 0
    rows.append((name, n, acc, n_errors, mean_lat))

if rows:
    rows.sort(key=lambda r: -r[2])
    print(f"{'Model':<22} {'N':>5} {'Accuracy':>9} {'Errors':>7} {'Lat(s)':>8}")
    print("─" * 56)
    for name, n, acc, nerr, lat in rows:
        winner = " ◀ WINNER" if rows and rows[0][0] == name and len(rows) > 1 else ""
        print(f"{name:<22} {n:>5} {acc:>9.3f} {nerr:>7} {lat:>8.2f}{winner}")
    print()
    winner_name = rows[0][0]
    spec = MODEL_REGISTRY[winner_name]
    print(f"Selected cortex model: {winner_name}  ({spec.hf_id})")
    print("Next step: pin the HF commit hash in models.py → revision field.")
else:
    print("No bake-off results found yet.")

---
## Cell 5 — Propose a variant *(optional)*
Packages the current harness directory as a valid content-hashed variant bundle
using `write_bundle()`, stages it atomically, and writes a `.ready` sentinel so
the governor never reviews a half-synced proposal.

In [ ]:
import json, pathlib, shutil, time
from prometheus.common.variant import write_bundle

# ── Parameters ────────────────────────────────────────────────────────────────
VARIANT_NAME   = "g1"          # short label, no spaces
HARNESS_DIR    = pathlib.Path(f"{INSTALL_ROOT}/current/harness")
PROXY_SCORE    = None          # float if you have an evaluator score, else None
AUDIT_CKPT     = pathlib.Path(f"{EXCHANGE_DIR}/audit_checkpoint.json")

# ── Load audit claims (must cover AUDIT_SEEDS from governor policy) ──────────
claims_src = AUDIT_CKPT if AUDIT_CKPT.exists() else pathlib.Path(CHECKPOINT_PATH)
if not claims_src.exists():
    raise RuntimeError("No checkpoint found — evaluate AUDIT_SEEDS first.")
cp_state = json.loads(claims_src.read_text())
audit_claims = {
    str(seed): {k: bool(v) for k, v in cp_state["results"][str(seed)]["claims"].items()}
    for seed in AUDIT_SEEDS
    if str(seed) in cp_state.get("results", {})
}
if len(audit_claims) < len(AUDIT_SEEDS):
    print(
        f"WARNING: only {len(audit_claims)}/{len(AUDIT_SEEDS)} AUDIT_SEEDS present in "
        f"{claims_src.name}. Missing audit seeds score 0 on the governor."
    )

# ── Collect harness files ─────────────────────────────────────────────────────
harness_files: dict[str, str] = {}
if HARNESS_DIR.exists():
    for p in sorted(HARNESS_DIR.rglob("*")):
        if p.is_file() and not p.is_symlink():
            rel = f"harness/{p.relative_to(HARNESS_DIR).as_posix()}"
            harness_files[rel] = p.read_text(encoding="utf-8")
else:
    print(f"Warning: {HARNESS_DIR} not found — bundle will have no files.")

# ── Stage and publish bundle via write_bundle() ──────────────────────────────
proposals_root = pathlib.Path(EXCHANGE_DIR) / "proposals"
proposals_root.mkdir(parents=True, exist_ok=True)
staging_dir  = proposals_root / f".staging-{VARIANT_NAME}"
proposal_dir = proposals_root / VARIANT_NAME
if staging_dir.exists():
    shutil.rmtree(staging_dir)
if proposal_dir.exists():
    shutil.rmtree(proposal_dir)

manifest_fields = {
    "run_id":      RUN_ID,
    "generation":  gate.generation + 1,
    "parent_hash": gate.installed_hash,
    "model_pins":  MODEL_PINS,
    "created_at":  time.time(),
}
if PROXY_SCORE is not None:
    manifest_fields["proxy_score"] = PROXY_SCORE

bundle = write_bundle(staging_dir, manifest_fields, harness_files)
(staging_dir / "audit_claims.json").write_text(
    json.dumps(audit_claims, indent=2, sort_keys=True), encoding="utf-8"
)
staging_dir.rename(proposal_dir)
# Write .ready sentinel LAST so rclone/governor knows the upload is complete
(proposal_dir / ".ready").write_text(bundle.variant_hash + "\n", encoding="utf-8")

print(f"Bundle name   : {VARIANT_NAME}")
print(f"Variant hash  : {bundle.variant_hash}")
print(f"Generation    : {manifest_fields['generation']}")
print(f"Parent hash   : {gate.installed_hash[:16]}…")
print(f"Files         : {len(harness_files)}")
print(f"Audit claims  : {len(audit_claims)}/{len(AUDIT_SEEDS)} worlds")
print()
print("Bundle published. Check exchange/approvals/ after the next governor tick.")

---
## Cell 6 — Install an approved variant
After the governor writes an approval to `exchange/approvals/<name>.json`,
call `gate.install()` to atomically swap the harness into place.

In [ ]:
VARIANT_TO_INSTALL = "g1"    # must match Cell 5's VARIANT_NAME

proposal_dir = pathlib.Path(EXCHANGE_DIR) / "proposals" / VARIANT_TO_INSTALL
try:
    dest = gate.install(proposal_dir)
    print(f"Installed     : generation={gate.generation}, dest={dest}")
    print(f"Installed hash: {gate.installed_hash}")
except InstallRefused as e:
    print(f"Install refused: {e}")
    raise
except Halted as e:
    print(f"Halted before install: {e}")
    raise